# L1 — Rapport de Profiling : E-Commerce Shipping Dataset
## Projet 2 — Segmentation RFM, CLV & Optimisation E-Commerce
---
**Objectif** : Analyser exhaustivement la qualité du dataset logistique/promotionnel E-Commerce Shipping.  
**Livrables** : Taux de nullité, outliers, doublons, distributions et anomalies pour 100% des colonnes.

> **Sous-thèmes concernés** : ST1 (funnel conversion), ST2 (satisfaction client / CLV), ST4 (promotions, livraison, marge)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Installation (à exécuter une seule fois)
import subprocess
subprocess.run(['pip', 'install', 'ydata-profiling', 'plotly', 'seaborn', '-q'])
print(' Librairies installées')

 Librairies installées


In [3]:
import os

base = "/content/drive/MyDrive/Projet_RFM_CLV_Ecommerce"
print(os.listdir(base))

['Profiling', 'Data', 'Notebooks', 'DBT_Gold', 'ETL_Bronze_Silver']


In [4]:
import os

for root, dirs, files in os.walk("/content/drive/MyDrive/Projet_RFM_CLV_Ecommerce/Data"):
    for file in files:
        if "shipping" in file.lower():
            print(os.path.join(root, file))

/content/drive/MyDrive/Projet_RFM_CLV_Ecommerce/Data/silver/ecommerce_shipping_silver.parquet
/content/drive/MyDrive/Projet_RFM_CLV_Ecommerce/Data/raw/ecommerce_shipping.csv
/content/drive/MyDrive/Projet_RFM_CLV_Ecommerce/Data/gold/csv_exports/stg_shipping.csv


## 0. Configuration & Imports

In [5]:
# ── Installation des dépendances (à exécuter une seule fois) ──
# !pip install ydata-profiling pandas plotly seaborn

import os
import json
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import seaborn as sns
import matplotlib.pyplot as plt
from ydata_profiling import ProfileReport
from IPython.display import display

pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.4f}'.format)

DATASET_PATH = '/content/drive/MyDrive/Projet_RFM_CLV_Ecommerce/Data/raw/ecommerce_shipping.csv'  # ← adapter au chemin réel dans votre Drive
REPORT_OUTPUT = '/content/drive/MyDrive/Projet_RFM_CLV_Ecommerce//Profiling/L1_Shipping_profile_report.html'

PALETTE = px.colors.qualitative.Set2
print(' Imports OK')

 Imports OK


## 1. Chargement du Dataset

In [6]:
df_raw = pd.read_csv(DATASET_PATH, encoding='utf-8')

print(f'Chargement : {df_raw.shape[0]:,} lignes × {df_raw.shape[1]} colonnes')
print(f'\nColonnes détectées : {list(df_raw.columns)}')
df_raw.head(5)

Chargement : 10,999 lignes × 12 colonnes

Colonnes détectées : ['ID', 'Warehouse_block', 'Mode_of_Shipment', 'Customer_care_calls', 'Customer_rating', 'Cost_of_the_Product', 'Prior_purchases', 'Product_importance', 'Gender', 'Discount_offered', 'Weight_in_gms', 'Reached.on.Time_Y.N']


,ID,Warehouse_block,Mode_of_Shipment,Customer_care_calls,Customer_rating,Cost_of_the_Product,Prior_purchases,Product_importance,Gender,Discount_offered,Weight_in_gms,Reached.on.Time_Y.N
0,1,D,Flight,4,2,177,3,low,F,44,1233,1
1,2,F,Flight,4,5,216,2,low,M,59,3088,1
2,3,A,Flight,2,2,183,4,low,M,48,3374,1
3,4,B,Flight,3,3,176,4,medium,M,10,1177,1
4,5,C,Flight,2,2,184,3,medium,F,46,2484,1


## 2. Vue d'ensemble — Métriques Globales

In [7]:
total_rows = len(df_raw)
total_cols = df_raw.shape[1]
total_cells = total_rows * total_cols
total_nulls = df_raw.isnull().sum().sum()
total_dupes = df_raw.duplicated().sum()

summary = {
    'Total lignes'            : f'{total_rows:,}',
    'Total colonnes'          : total_cols,
    'Taux nullité global'     : f'{total_nulls/total_cells*100:.2f}%',
    'Lignes dupliquées'       : f'{total_dupes:,} ({total_dupes/total_rows*100:.2f}%)',
    'Types de livraison'      : df_raw['Mode_of_Shipment'].nunique() if 'Mode_of_Shipment' in df_raw.columns else 'N/A',
    'Niveaux importance produit': df_raw['Product_importance'].nunique() if 'Product_importance' in df_raw.columns else 'N/A',
    'Livraisons à temps (=1)' : f"{df_raw['Reached.on.Time_Y.N'].sum():,}" if 'Reached.on.Time_Y.N' in df_raw.columns else 'N/A',
}

pd.DataFrame.from_dict(summary, orient='index', columns=['Valeur']).style.set_caption(' Vue Globale E-Commerce Shipping')

,Valeur
Total lignes,"10,999"
Total colonnes,12
Taux nullité global,0.00%
Lignes dupliquées,0 (0.00%)
Types de livraison,3
Niveaux importance produit,3
Livraisons à temps (=1),"6,563"


## 3. Analyse de Nullité — 100% des Colonnes

In [9]:
null_df = pd.DataFrame({
    'Colonne': df_raw.columns,
    'Type': df_raw.dtypes.values,
    'Nulls': df_raw.isnull().sum().values,
    'Taux_Null_%': (df_raw.isnull().sum() / len(df_raw) * 100).values,
    'Valeurs_uniques': df_raw.nunique().values,
    'Exemple': [df_raw[c].dropna().iloc[0] if df_raw[c].dropna().shape[0] > 0 else 'N/A' for c in df_raw.columns]
})
null_df['Criticité'] = null_df['Taux_Null_%'].apply(
    lambda x: ' Critique (>20%)' if x > 20 else (' Élevé (5-20%)' if x > 5 else (' Modéré (1-5%)' if x > 1 else ' OK (<1%)'))
)

display(null_df.style
    .format({'Taux_Null_%': '{:.2f}%'})
    .background_gradient(subset=['Taux_Null_%'], cmap='RdYlGn_r')
    .set_caption(' Analyse Nullité — Toutes les colonnes Shipping'))

fig = px.bar(
    null_df.sort_values('Taux_Null_%', ascending=True),
    x='Taux_Null_%', y='Colonne', orientation='h',
    color='Taux_Null_%', color_continuous_scale='RdYlGn_r',
    title='Taux de Nullité par Colonne — Shipping Dataset'
)
fig.update_layout(height=500, showlegend=False)
fig.show()

,Colonne,Type,Nulls,Taux_Null_%,Valeurs_uniques,Exemple,Criticité
0,ID,int64,0,0.00%,10999,1,OK (<1%)
1,Warehouse_block,object,0,0.00%,5,D,OK (<1%)
2,Mode_of_Shipment,object,0,0.00%,3,Flight,OK (<1%)
3,Customer_care_calls,int64,0,0.00%,6,4,OK (<1%)
4,Customer_rating,int64,0,0.00%,5,2,OK (<1%)
5,Cost_of_the_Product,int64,0,0.00%,215,177,OK (<1%)
6,Prior_purchases,int64,0,0.00%,8,3,OK (<1%)
7,Product_importance,object,0,0.00%,3,low,OK (<1%)
8,Gender,object,0,0.00%,2,F,OK (<1%)
9,Discount_offered,int64,0,0.00%,65,44,OK (<1%)


## 4. Analyse Colonne par Colonne

### 4.1 — Mode_of_Shipment

In [10]:
col = 'Mode_of_Shipment'
s = df_raw[col]

print(f'=== {col} ===')
print(f'  Nulls          : {s.isnull().sum():,}')
print(f'  Modes uniques  : {s.unique()}')

vc = s.value_counts().reset_index()
vc.columns = ['Mode', 'Count']
vc['Pct'] = vc['Count'] / len(s) * 100
display(vc.style.format({'Pct': '{:.1f}%'}))

fig = px.pie(vc, values='Count', names='Mode',
             title='Mode_of_Shipment — Répartition',
             color_discrete_sequence=PALETTE)
fig.show()

=== Mode_of_Shipment ===
  Nulls          : 0
  Modes uniques  : ['Flight' 'Ship' 'Road']


,Mode,Count,Pct
0,Ship,7462,67.8%
1,Flight,1777,16.2%
2,Road,1760,16.0%


### 4.2 — Discount_offered

In [11]:
col = 'Discount_offered'
s = df_raw[col]

print(f'=== {col} ===')
print(f'  Nulls          : {s.isnull().sum():,}')
print(f'  Min / Max      : {s.min():.1f}% / {s.max():.1f}%')
print(f'  Médiane        : {s.median():.1f}%')
print(f'  Remises = 0%   : {(s == 0).sum():,} ({(s == 0).mean()*100:.1f}%)')
print(f'  Remises > 50%  : {(s > 50).sum():,}')

display(s.describe().to_frame().style.format('{:.2f}').set_caption('Stats Discount_offered (%)'))

# Catégorisation par tranche (cf. CDC Section 6.4)
bins = [-0.1, 0, 10, 25, 50, 100]
labels = ['0% (Prix plein)', '1-10% (Légère)', '11-25% (Significative)', '26-50% (Forte)', '>50% (Liquidation)']
df_raw['discount_tranche'] = pd.cut(s, bins=bins, labels=labels)
tranche_counts = df_raw['discount_tranche'].value_counts().sort_index().reset_index()
tranche_counts.columns = ['Tranche', 'Count']

fig = px.bar(tranche_counts, x='Tranche', y='Count',
             title='Discount_offered — Répartition par tranche',
             color='Count', color_continuous_scale='Oranges',
             text='Count')
fig.update_layout(height=400, xaxis_tickangle=-20)
fig.show()

# Histogramme détaillé
fig2 = px.histogram(s, nbins=60, title='Distribution détaillée Discount_offered',
                    labels={'value': 'Remise (%)'}, color_discrete_sequence=[PALETTE[1]])
fig2.update_layout(height=360)
fig2.show()

=== Discount_offered ===
  Nulls          : 0
  Min / Max      : 1.0% / 65.0%
  Médiane        : 7.0%
  Remises = 0%   : 0 (0.0%)
  Remises > 50%  : 733


,Discount_offered
count,10999.00
mean,13.37
std,16.21
min,1.00
25%,4.00
50%,7.00
75%,10.00
max,65.00


### 4.3 — Cost_of_the_Product

In [12]:
col = 'Cost_of_the_Product'
s = df_raw[col]

q1, q3 = s.quantile(0.25), s.quantile(0.75)
iqr = q3 - q1
outliers = s[(s < q1 - 1.5*iqr) | (s > q3 + 1.5*iqr)]

print(f'=== {col} ===')
print(f'  Nulls          : {s.isnull().sum():,}')
print(f'  Min / Max      : {s.min():,.0f} / {s.max():,.0f}')
print(f'  Médiane        : {s.median():,.0f}')
print(f'  Outliers IQR×1.5 : {len(outliers):,}')
print(f'  Coûts <= 0     : {(s <= 0).sum():,}')

display(s.describe().to_frame().style.format('{:,.2f}').set_caption('Stats Cost_of_the_Product'))

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Histogramme Cost', 'Boxplot Cost'])
fig.add_trace(go.Histogram(x=s, nbinsx=60, name='Cost', marker_color=PALETTE[2]), row=1, col=1)
fig.add_trace(go.Box(y=s, name='Boxplot', boxpoints='outliers', marker_color=PALETTE[3]), row=1, col=2)
fig.update_layout(title='Cost_of_the_Product — Distribution', height=400, showlegend=False)
fig.show()

=== Cost_of_the_Product ===
  Nulls          : 0
  Min / Max      : 96 / 310
  Médiane        : 214
  Outliers IQR×1.5 : 0
  Coûts <= 0     : 0


,Cost_of_the_Product
count,"10,999.00"
mean,210.20
std,48.06
min,96.00
25%,169.00
50%,214.00
75%,251.00
max,310.00


### 4.4 — Weight_in_gms

In [13]:
col = 'Weight_in_gms'
s = df_raw[col]

q1, q3 = s.quantile(0.25), s.quantile(0.75)
iqr = q3 - q1
outliers = s[(s < q1 - 1.5*iqr) | (s > q3 + 1.5*iqr)]

print(f'=== {col} ===')
print(f'  Nulls          : {s.isnull().sum():,}')
print(f'  Min / Max      : {s.min():,}g / {s.max():,}g')
print(f'  Médiane        : {s.median():,.0f}g ({s.median()/1000:.2f}kg)')
print(f'  Poids <= 0     : {(s <= 0).sum():,} (anomalie)')
print(f'  Outliers IQR×1.5 : {len(outliers):,}')

display(s.describe().to_frame().style.format('{:,.0f}').set_caption('Stats Weight_in_gms'))

# Catégories de poids pour frais de port
bins_w = [0, 1000, 2000, 4000, 6000, 99999]
labels_w = ['<1kg', '1-2kg', '2-4kg', '4-6kg', '>6kg']
df_raw['weight_cat'] = pd.cut(s, bins=bins_w, labels=labels_w)
w_cat = df_raw['weight_cat'].value_counts().sort_index().reset_index()
w_cat.columns = ['Catégorie poids', 'Count']

fig = px.bar(w_cat, x='Catégorie poids', y='Count',
             title='Weight_in_gms — Répartition par catégorie de poids',
             color='Count', color_continuous_scale='Blues', text='Count')
fig.update_layout(height=380)
fig.show()

=== Weight_in_gms ===
  Nulls          : 0
  Min / Max      : 1,001g / 7,846g
  Médiane        : 4,149g (4.15kg)
  Poids <= 0     : 0 (anomalie)
  Outliers IQR×1.5 : 0


,Weight_in_gms
count,"10,999"
mean,"3,634"
std,"1,635"
min,"1,001"
25%,"1,840"
50%,"4,149"
75%,"5,050"
max,"7,846"


### 4.5 — Reached.on.Time_Y.N

In [14]:
col = 'Reached.on.Time_Y.N'
s = df_raw[col]

print(f'=== {col} ===')
print(f'  Nulls          : {s.isnull().sum():,}')
print(f'  Valeurs uniques: {s.unique()}')
on_time_rate = (s == 1).mean() * 100
late_rate = (s == 0).mean() * 100
print(f'  Livraisons dans les délais (1) : {(s == 1).sum():,} ({on_time_rate:.1f}%)')
print(f'  Livraisons en retard (0)       : {(s == 0).sum():,} ({late_rate:.1f}%)')
print(f'  Valeurs hors 0/1 (anomalie)    : {(~s.isin([0, 1])).sum():,}')

# Visualisation par mode de livraison
if 'Mode_of_Shipment' in df_raw.columns:
    on_time_by_mode = df_raw.groupby('Mode_of_Shipment')[col].mean().reset_index()
    on_time_by_mode.columns = ['Mode', 'Taux_ponctualite']
    on_time_by_mode['Taux_ponctualite_pct'] = on_time_by_mode['Taux_ponctualite'] * 100

    fig = px.bar(on_time_by_mode, x='Mode', y='Taux_ponctualite_pct',
                 title='Taux de ponctualité par mode de livraison',
                 color='Mode', color_discrete_sequence=PALETTE,
                 labels={'Taux_ponctualite_pct': 'Taux ponctualité (%)'},
                 text=on_time_by_mode['Taux_ponctualite_pct'].round(1).astype(str) + '%')
    fig.update_layout(height=400, showlegend=False)
    fig.show()

=== Reached.on.Time_Y.N ===
  Nulls          : 0
  Valeurs uniques: [1 0]
  Livraisons dans les délais (1) : 6,563 (59.7%)
  Livraisons en retard (0)       : 4,436 (40.3%)
  Valeurs hors 0/1 (anomalie)    : 0


### 4.6 — Customer_rating

In [15]:
col = 'Customer_rating'
s = df_raw[col]

print(f'=== {col} ===')
print(f'  Nulls          : {s.isnull().sum():,}')
print(f'  Valeurs uniques: {sorted(s.unique())}')
print(f'  Hors [1-5]     : {(~s.isin([1,2,3,4,5])).sum():,} (anomalies)')
print(f'  Moyenne        : {s.mean():.3f}')
print(f'  Médiane        : {s.median():.1f}')

vc = s.value_counts().sort_index().reset_index()
vc.columns = ['Note', 'Count']
vc['Pct'] = vc['Count'] / len(s) * 100

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Distribution des notes', 'Note vs Ponctualité'])
fig.add_trace(
    go.Bar(x=vc['Note'].astype(str), y=vc['Count'],
           marker_color=PALETTE[:len(vc)], text=vc['Count'], textposition='outside'),
    row=1, col=1
)

if 'Reached.on.Time_Y.N' in df_raw.columns:
    note_ponct = df_raw.groupby('Customer_rating')['Reached.on.Time_Y.N'].mean().reset_index()
    fig.add_trace(
        go.Scatter(x=note_ponct['Customer_rating'],
                   y=note_ponct['Reached.on.Time_Y.N'] * 100,
                   mode='lines+markers', name='Ponctualité (%)',
                   marker_color=PALETTE[4]),
        row=1, col=2
    )

fig.update_layout(title='Customer_rating — Distribution & Corrélations', height=420, showlegend=False)
fig.show()

=== Customer_rating ===
  Nulls          : 0
  Valeurs uniques: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
  Hors [1-5]     : 0 (anomalies)
  Moyenne        : 2.991
  Médiane        : 3.0


### 4.7 — Prior_purchases

In [16]:
col = 'Prior_purchases'
s = df_raw[col]

q1, q3 = s.quantile(0.25), s.quantile(0.75)
iqr = q3 - q1
outliers = s[(s < q1 - 1.5*iqr) | (s > q3 + 1.5*iqr)]

print(f'=== {col} === (input CLV — ST2)')
print(f'  Nulls          : {s.isnull().sum():,}')
print(f'  Min / Max      : {s.min():,} / {s.max():,}')
print(f'  Médiane        : {s.median():.1f}')
print(f'  Achat unique (=0) : {(s == 0).sum():,}')
print(f'  Outliers IQR×1.5 : {len(outliers):,}')

display(s.describe().to_frame().style.format('{:.2f}').set_caption('Stats Prior_purchases'))

vc = s.value_counts().sort_index().reset_index()
vc.columns = ['Prior_purchases', 'Count']
fig = px.bar(vc, x='Prior_purchases', y='Count',
             title='Distribution Prior_purchases (input CLV)',
             color='Count', color_continuous_scale='Greens', text='Count')
fig.update_layout(height=380)
fig.show()

=== Prior_purchases === (input CLV — ST2)
  Nulls          : 0
  Min / Max      : 2 / 10
  Médiane        : 3.0
  Achat unique (=0) : 0
  Outliers IQR×1.5 : 1,003


,Prior_purchases
count,10999.00
mean,3.57
std,1.52
min,2.00
25%,3.00
50%,3.00
75%,4.00
max,10.00


### 4.8 — Customer_care_calls

In [17]:
col = 'Customer_care_calls'
s = df_raw[col]

print(f'=== {col} === (proxy insatisfaction — ST4)')
print(f'  Nulls          : {s.isnull().sum():,}')
print(f'  Min / Max      : {s.min():,} / {s.max():,}')
print(f'  Médiane        : {s.median():.1f}')
print(f'  0 appel        : {(s == 0).sum():,} ({(s == 0).mean()*100:.1f}%)')
print(f'  5+ appels      : {(s >= 5).sum():,}')

display(s.describe().to_frame().style.format('{:.2f}').set_caption('Stats Customer_care_calls'))

# Corrélation appels SAV vs rating
if 'Customer_rating' in df_raw.columns:
    corr_calls_rating = df_raw.groupby(col)['Customer_rating'].mean().reset_index()
    corr_calls_rating.columns = ['Nb_appels', 'Rating_moyen']
    fig = px.line(corr_calls_rating, x='Nb_appels', y='Rating_moyen',
                  title='Appels SAV vs Note moyenne — Corrélation',
                  markers=True,
                  labels={'Nb_appels': "Nb d'appels SAV", 'Rating_moyen': 'Note client moyenne'})
    fig.update_layout(height=380)
    fig.show()

=== Customer_care_calls === (proxy insatisfaction — ST4)
  Nulls          : 0
  Min / Max      : 2 / 7
  Médiane        : 4.0
  0 appel        : 0 (0.0%)
  5+ appels      : 3,587


,Customer_care_calls
count,10999.00
mean,4.05
std,1.14
min,2.00
25%,3.00
50%,4.00
75%,5.00
max,7.00


### 4.9 — Product_importance

In [18]:
col = 'Product_importance'
s = df_raw[col]

print(f'=== {col} ===')
print(f'  Nulls          : {s.isnull().sum():,}')
print(f'  Niveaux uniques: {s.unique()}')

vc = s.value_counts().reset_index()
vc.columns = ['Importance', 'Count']
vc['Pct'] = vc['Count'] / len(s) * 100
display(vc.style.format({'Pct': '{:.1f}%'}))

# Importance vs livraison dans les délais
if 'Reached.on.Time_Y.N' in df_raw.columns:
    imp_ponct = df_raw.groupby(col)['Reached.on.Time_Y.N'].mean().reset_index()
    imp_ponct.columns = ['Importance', 'Taux_ponctualite']
    imp_ponct['Pct'] = imp_ponct['Taux_ponctualite'] * 100

    fig = px.bar(imp_ponct, x='Importance', y='Pct',
                 title='Product_importance vs Taux de ponctualité livraison',
                 color='Importance', color_discrete_sequence=PALETTE,
                 text=imp_ponct['Pct'].round(1).astype(str) + '%',
                 labels={'Pct': 'Taux ponctualité (%)'})
    fig.update_layout(height=380, showlegend=False)
    fig.show()

=== Product_importance ===
  Nulls          : 0
  Niveaux uniques: ['low' 'medium' 'high']


,Importance,Count,Pct
0,low,5297,48.2%
1,medium,4754,43.2%
2,high,948,8.6%


## 5. Détection des Doublons

In [19]:
dupes_strict = df_raw.duplicated()
print(f'Doublons stricts : {dupes_strict.sum():,} ({dupes_strict.mean()*100:.3f}%)')

if dupes_strict.sum() > 0:
    print('\nExemples de doublons :')
    display(df_raw[dupes_strict].head(5))
else:
    print('✅ Aucun doublon strict détecté.')

Doublons stricts : 0 (0.000%)
✅ Aucun doublon strict détecté.


## 6. Analyse des Corrélations

In [20]:
# Sélection des colonnes numériques pour la matrice de corrélation
numeric_cols = df_raw.select_dtypes(include=[np.number]).columns.tolist()
print(f'Colonnes numériques : {numeric_cols}')

if len(numeric_cols) >= 2:
    corr_matrix = df_raw[numeric_cols].corr()

    fig = px.imshow(
        corr_matrix,
        color_continuous_scale='RdBu_r',
        zmin=-1, zmax=1,
        title='Matrice de Corrélation — Variables Numériques Shipping',
        text_auto='.2f'
    )
    fig.update_layout(height=520)
    fig.show()

    print('\n⚡ Corrélations > |0.3| à analyser :')
    corr_flat = corr_matrix.unstack().reset_index()
    corr_flat.columns = ['Var1', 'Var2', 'Correlation']
    strong = corr_flat[(abs(corr_flat['Correlation']) > 0.3) &
                       (corr_flat['Var1'] != corr_flat['Var2'])].drop_duplicates(subset=['Correlation'])
    display(strong.sort_values('Correlation', key=abs, ascending=False).head(15))

Colonnes numériques : ['ID', 'Customer_care_calls', 'Customer_rating', 'Cost_of_the_Product', 'Prior_purchases', 'Discount_offered', 'Weight_in_gms', 'Reached.on.Time_Y.N']



⚡ Corrélations > |0.3| à analyser :


,Var1,Var2,Correlation
5,ID,Discount_offered,-0.5983
7,ID,Reached.on.Time_Y.N,-0.4118
47,Discount_offered,Reached.on.Time_Y.N,0.3971
46,Discount_offered,Weight_in_gms,-0.3761
11,Customer_care_calls,Cost_of_the_Product,0.3232


## 7. Analyse des Anomalies Métier (ST4)

In [22]:
print('=== Anomalies Métier ST4 — Promotions & Livraison ===')

# 1. Remises extrêmes
if 'Discount_offered' in df_raw.columns:
    extreme_disc = df_raw[df_raw['Discount_offered'] > 65]
    print(f'\n[1] Remises > 65%  : {len(extreme_disc):,} commandes')

# 2. Coût produit vs remise (remise > coût ?)
if all(c in df_raw.columns for c in ['Cost_of_the_Product', 'Discount_offered']):
    # Estimation revenue approximatif
    # Marge brute avant remise
    marge_estimate = df_raw['Cost_of_the_Product'] * (1 - df_raw['Discount_offered'] / 100)
    negative_margin = (marge_estimate < 0).sum()
    print(f'[2] Commandes à marge estimée négative : {negative_margin:,}')

# 3. Retards vs rating faible
if all(c in df_raw.columns for c in ['Reached.on.Time_Y.N', 'Customer_rating']):
    late_low_rating = df_raw[(df_raw['Reached.on.Time_Y.N'] == 0) & (df_raw['Customer_rating'] <= 2)]
    print(f'[3] Retards + note ≤ 2 : {len(late_low_rating):,} ({len(late_low_rating)/len(df_raw)*100:.1f}%)')

# 4. Poids nul ou négatif
if 'Weight_in_gms' in df_raw.columns:
    zero_weight = df_raw[df_raw['Weight_in_gms'] <= 0]
    print(f'[4] Poids <= 0g     : {len(zero_weight):,}')

print('\n Analyse anomalies métier terminée')

=== Anomalies Métier ST4 — Promotions & Livraison ===

[1] Remises > 65%  : 0 commandes
[2] Commandes à marge estimée négative : 0
[3] Retards + note ≤ 2 : 1,814 (16.5%)
[4] Poids <= 0g     : 0

 Analyse anomalies métier terminée


## 8. Rapport ydata-profiling (Automatique)

In [24]:
print(' Génération du rapport ydata-profiling Shipping...')

profile = ProfileReport(
    df_raw,
    title='L1 — Profiling E-Commerce Shipping',
    explorative=True,
    correlations={'pearson': {'calculate': True},
                  'spearman': {'calculate': False},
                  'kendall': {'calculate': False},
                  'phi_k': {'calculate': False}},
    missing_diagrams={'bar': True, 'matrix': True, 'heatmap': True},
    samples={'head': 10, 'tail': 5},
    progress_bar=True
)

os.makedirs(os.path.dirname(REPORT_OUTPUT), exist_ok=True)
profile.to_file(REPORT_OUTPUT)
print(f' Rapport exporté : {REPORT_OUTPUT}')

 Génération du rapport ydata-profiling Shipping...


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 14/14 [00:00<00:00, 37.79it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

 Rapport exporté : /content/drive/MyDrive/Projet_RFM_CLV_Ecommerce//Profiling/L1_Shipping_profile_report.html


## 9. Synthèse — Qualité du Dataset Shipping

In [25]:
print("""
╔═══════════════════════════════════════════════════════════════╗
║        SYNTHÈSE PROFILING — E-Commerce Shipping              ║
╠═══════════════════════════════════════════════════════════════╣
║ CONSTAT                        │ USAGE / ACTION               ║
╠═══════════════════════════════════════════════════════════════╣
║ Mode_of_Shipment (3 modes)     │ Analyse ST4 livraison        ║
║ Discount_offered (0-65%+)      │ Calcul break-even promo ST4  ║
║ Cost_of_the_Product            │ Calcul marge brute ST4       ║
║ Weight_in_gms                  │ Grille frais de port ST4     ║
║ Reached.on.Time_Y.N (binaire)  │ KPI ponctualité ST1 & ST4   ║
║ Customer_rating (1-5)          │ Satisfaction ST2 CLV/RFM     ║
║ Prior_purchases                │ Historique client CLV ST2    ║
║ Customer_care_calls            │ Proxy insatisfaction ST4     ║
║ Product_importance (L/M/H)     │ Priorité traitement ST4      ║
╠═══════════════════════════════════════════════════════════════╣
║   Pas de CustomerID dans ce dataset → jointure indirecte   ║
║   Vérifier la jointure avec UCI sur InvoiceNo / OrderID    ║
╚═══════════════════════════════════════════════════════════════╝
""")

# Export métriques JSON
metrics = {
    'dataset': 'E-Commerce Shipping',
    'total_rows': int(total_rows),
    'total_cols': int(total_cols),
    'null_rate_global_pct': round(total_nulls / total_cells * 100, 2),
    'duplicate_rows': int(total_dupes),
    'null_by_column': {c: int(df_raw[c].isnull().sum()) for c in df_raw.columns},
    'unique_shipment_modes': list(df_raw['Mode_of_Shipment'].unique()) if 'Mode_of_Shipment' in df_raw.columns else [],
    'discount_range': {
        'min': float(df_raw['Discount_offered'].min()) if 'Discount_offered' in df_raw.columns else None,
        'max': float(df_raw['Discount_offered'].max()) if 'Discount_offered' in df_raw.columns else None,
    }
}

metrics_path = '../Profiling/L1_Shipping_metrics.json'
os.makedirs(os.path.dirname(metrics_path), exist_ok=True)
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

print(f' Métriques exportées : {metrics_path}')
print('\n Notebook L1 Shipping — Profiling terminé.')


╔═══════════════════════════════════════════════════════════════╗
║        SYNTHÈSE PROFILING — E-Commerce Shipping              ║
╠═══════════════════════════════════════════════════════════════╣
║ CONSTAT                        │ USAGE / ACTION               ║
╠═══════════════════════════════════════════════════════════════╣
║ Mode_of_Shipment (3 modes)     │ Analyse ST4 livraison        ║
║ Discount_offered (0-65%+)      │ Calcul break-even promo ST4  ║
║ Cost_of_the_Product            │ Calcul marge brute ST4       ║
║ Weight_in_gms                  │ Grille frais de port ST4     ║
║ Reached.on.Time_Y.N (binaire)  │ KPI ponctualité ST1 & ST4   ║
║ Customer_rating (1-5)          │ Satisfaction ST2 CLV/RFM     ║
║ Prior_purchases                │ Historique client CLV ST2    ║
║ Customer_care_calls            │ Proxy insatisfaction ST4     ║
║ Product_importance (L/M/H)     │ Priorité traitement ST4      ║
╠═══════════════════════════════════════════════════════════════╣
║   Pas de 